# Exploring the Pipedrive extract

A question-led walkthrough of the analysis behind the monthly sales funnel. Each question pairs the existing SQL with results, interpretation and a modeling decision. This is a retrospective presentation of the work, rather than a chronological lab log.

**Reading:** outputs are included; no database is needed to review them in Jupyter or Colab. **Rerunning:** first load the project and complete `dbt build` as described in the README. Use a Python kernel with `psycopg2-binary` installed. Colab needs a reachable Postgres host; its `localhost` is not your computer. Connection settings below use environment variables, with the assignment's local defaults.

The first three questions query raw sources. Later questions also inspect the completed models and are explicitly validation of modeling choices. SQL is copied unchanged from the named `analyses/sql/*.sql` files; the small helper resolves this project's `source()` and `ref()` names, without reimplementing business logic. Saved outputs describe one database snapshot, not a live monitoring view.


In [1]:
import os
import re
import html
from datetime import datetime, timezone
import psycopg2
from IPython.display import display, HTML

conn = psycopg2.connect(
    host=os.getenv("PGHOST", "localhost"),
    port=os.getenv("PGPORT", "5432"),
    dbname=os.getenv("PGDATABASE", "postgres"),
    user=os.getenv("PGUSER", "admin"),
    password=os.getenv("PGPASSWORD", "admin"),
    connect_timeout=10,
)
conn.set_session(readonly=True, isolation_level="REPEATABLE READ", autocommit=False)

def resolve_sql(sql):
    # Only the source/ref expressions used by these saved analyses are supported.
    def source(match):
        if match.group(1) != "pipedrive":
            raise ValueError("Unexpected source")
        return '"public"."' + match.group(2) + '"'

    def ref(match):
        name = match.group(1)
        if name.startswith("ods_"):
            schema = "public_pipedrive_ods"
        elif name.startswith("rep_"):
            schema = "public_pipedrive_analytics"
        elif name.startswith(("dim_", "fct_")) or name == "funnel_steps":
            schema = "public_pipedrive_core"
        else:
            raise ValueError("Unexpected model: " + name)
        return '"' + schema + '"."' + name + '"'

    sql = re.sub(r"\{\{\s*source\('([a-z_]+)',\s*'([a-z_]+)'\)\s*\}\}", source, sql)
    sql = re.sub(r"\{\{\s*ref\('([a-z_]+)'\)\s*\}\}", ref, sql)
    if "{{" in sql or "{%" in sql:
        raise ValueError("Unsupported dbt template expression")
    return sql

def run_analysis(sql):
    with conn.cursor() as cursor:
        cursor.execute(resolve_sql(sql))
        columns = [column.name for column in cursor.description]
        rows = cursor.fetchall()
    heading = "".join("<th>" + html.escape(c) + "</th>" for c in columns)
    body = "".join("<tr>" + "".join("<td>" + html.escape(str(v) if v is not None else "NULL") + "</td>" for v in row) + "</tr>" for row in rows)
    display(HTML('<div style="overflow-x:auto"><table><thead><tr>' + heading + '</tr></thead><tbody>' + body + '</tbody></table></div>'))
    print(f"{len(rows)} result rows")

with conn.cursor() as cursor:
    cursor.execute("SELECT current_database(), transaction_timestamp(), version()")
    database, captured_at, version = cursor.fetchone()
print(f"Database: {database}; snapshot: {captured_at.isoformat()}; {version.split(',')[0]}")


Database: postgres; snapshot: 2026-11-05T10:37:56.697241+00:00; PostgreSQL 18.6 (Debian 18.6-1.pgdg13+2) on x86_64-pc-linux-gnu


## 1. What data is actually available?

**Scope:** Raw-source exploration. SQL: `sql/source_profile.sql`.

In [2]:
sql = r"""
-- Question: how big is each raw source, what do its keys look like, and what dates does it cover?
-- Backs: the source profile in docs/technical_documentation.md.
-- Result on this extract: 15,406 deal changes, 4,579 activities, 1,787 users.
with deal_changes as (

    select * from {{ source('pipedrive', 'deal_changes') }}

),

activity as (

    select * from {{ source('pipedrive', 'activity') }}

),

users as (

    select * from {{ source('pipedrive', 'users') }}

),

calls_per_deal as (

    select
        deal_id,
        count(*) as calls,
        count(*) filter (where type = 'meeting') as sales_call_1,
        count(*) filter (where type = 'sc_2') as sales_call_2
    from activity
    group by 1

)

select 'deal_changes rows' as metric, count(*)::text as value from deal_changes
union all select 'deal_changes deals', count(distinct deal_id)::text from deal_changes
union all select 'deal_changes first date', min(change_time)::date::text from deal_changes
union all select 'deal_changes last date', max(change_time)::date::text from deal_changes
union all select 'deal_changes stage rows', count(*)::text from deal_changes where changed_field_key = 'stage_id'
union all select 'first stage row', min(change_time)::date::text from deal_changes where changed_field_key = 'stage_id'
union all select 'last stage row', max(change_time)::date::text from deal_changes where changed_field_key = 'stage_id'
union all select 'last deal creation', max(change_time)::date::text from deal_changes where changed_field_key = 'add_time'
union all select 'last lost reason', max(change_time)::date::text from deal_changes where changed_field_key = 'lost_reason'
union all select 'activity rows', count(*)::text from activity
union all select 'activity distinct ids', count(distinct activity_id)::text from activity
union all select 'activity distinct deal ids', count(distinct deal_id)::text from activity
union all select 'activity first due date', min(due_to)::date::text from activity
union all select 'activity last due date', max(due_to)::date::text from activity
union all select 'users rows', count(*)::text from users
union all select 'users first modified', min(modified)::date::text from users
union all select 'users last modified', max(modified)::date::text from users
union all select 'stages rows', count(*)::text from {{ source('pipedrive', 'stages') }}
union all select 'activity_types rows', count(*)::text from {{ source('pipedrive', 'activity_types') }}
union all select 'fields rows', count(*)::text from {{ source('pipedrive', 'fields') }}
union all select 'deal ids with one call', count(*)::text from calls_per_deal where calls = 1
union all select 'deal ids with two calls', count(*)::text from calls_per_deal where calls = 2
union all select 'deal ids with both sales calls', count(*)::text from calls_per_deal where sales_call_1 > 0 and sales_call_2 > 0
union all select 'deal ids with two of one sales call', count(*)::text from calls_per_deal where sales_call_1 > 1 or sales_call_2 > 1
"""
run_analysis(sql)


metric,value
deal_changes rows,15406
deal_changes deals,1995
deal_changes first date,2024-01-01
deal_changes last date,2025-03-11
deal_changes stage rows,8906
first stage row,2024-01-08
last stage row,2025-02-24
last deal creation,2024-09-13
last lost reason,2025-03-11
activity rows,4579


24 result rows


**Interpretation.** The extract contains 15,406 deal changes, 4,579 activities and 1,787 users. Deal changes are a field-level event log, not one row per deal. Activities have their own identity and dates.

**Decision.** Standardize sources in ODS, then build deal entities and event facts at explicit grains. There is no supplied deal-status, won-value or actual call-completion timestamp: avoid inventing those measures.

## 2. Do the sources cover the same months?

**Scope:** Raw-source exploration. SQL: `sql/monthly_volumes.sql`.

In [3]:
sql = r"""
-- Question: how many deals, stage rows and calls does each month hold, on the raw sources?
-- Backs: the coverage statements in docs/technical_documentation.md.
-- Result on this extract: activities stop in September 2024, stage rows in February 2025.
with months as (

    select date_trunc('month', change_time)::date as month from {{ source('pipedrive', 'deal_changes') }}
    union
    select date_trunc('month', due_to)::date from {{ source('pipedrive', 'activity') }}

),

deal_rows as (

    select
        date_trunc('month', change_time)::date as month,
        count(*) filter (where changed_field_key = 'add_time') as deals_created,
        count(*) filter (where changed_field_key = 'stage_id') as stage_rows,
        count(*) filter (where changed_field_key = 'lost_reason') as lost_reasons
    from {{ source('pipedrive', 'deal_changes') }}
    group by 1

),

call_rows as (

    select
        date_trunc('month', due_to)::date as month,
        count(*) as calls,
        count(*) filter (where done and type in ('meeting', 'sc_2')) as done_sales_calls
    from {{ source('pipedrive', 'activity') }}
    group by 1

)

select
    months.month,
    coalesce(deal_rows.deals_created, 0) as deals_created,
    coalesce(deal_rows.stage_rows, 0) as stage_rows,
    coalesce(deal_rows.lost_reasons, 0) as lost_reasons,
    coalesce(call_rows.calls, 0) as calls,
    coalesce(call_rows.done_sales_calls, 0) as done_sales_calls
from months
left join deal_rows on months.month = deal_rows.month
left join call_rows on months.month = call_rows.month
order by months.month
"""
run_analysis(sql)


month,deals_created,stage_rows,lost_reasons,calls,done_sales_calls
2024-01-01,247,37,0,550,144
2024-02-01,205,333,8,506,123
2024-03-01,242,692,45,557,129
2024-04-01,244,870,130,492,129
2024-05-01,262,1011,190,532,126
2024-06-01,248,1108,207,551,130
2024-07-01,233,1137,248,580,153
2024-08-01,227,1150,251,577,129
2024-09-01,92,1017,268,234,65
2024-10-01,0,867,230,0,0


15 result rows


**Interpretation.** Activities and deal creation end in September 2024, recorded stage changes in February 2025, and lost reasons in March 2025. These are different observed endpoints, not proof of complete months or deliberate business cutoffs.

**Decision.** Use monthly output, expose source coverage, and distinguish no observed records from confirmed zero activity. The report ends at the last observed stage/activity month; lost-reason dates do not extend the reporting calendar.

## 3. Can calls be reliably joined to deal histories?

**Scope:** Raw-source exploration. SQL: `sql/call_deal_link_test.sql`.

In [4]:
sql = r"""
-- Question: can a call be tied to a deal? Joins deal_id, then activity_id, to the deal IDs, against what shuffled IDs
-- would give: IDs on one side * IDs on the other / size of the shared ID range.
-- Backs: the key assumption in docs/methodology_and_assumptions.md.
-- Result on this extract: 8 deal ID matches against 10 expected from shuffled IDs: too sparse to rely on.
with deal_ids as (

    select distinct deal_id from {{ source('pipedrive', 'deal_changes') }}

),

call_deal_ids as (

    select distinct deal_id from {{ source('pipedrive', 'activity') }}

),

activity_ids as (

    select distinct activity_id from {{ source('pipedrive', 'activity') }}

),

id_range as (

    select
        min(id) as lowest_id,
        max(id) as highest_id,
        (max(id) - min(id) + 1)::numeric as id_space
    from (
        select deal_id as id from deal_ids
        union all select deal_id from call_deal_ids
        union all select activity_id from activity_ids
    ) as ids

),

overlap_calls as (

    select
        activity.activity_id,
        activity.due_to,
        creation.created_at
    from {{ source('pipedrive', 'activity') }} as activity
    inner join (
        select deal_id, min(change_time) as created_at
        from {{ source('pipedrive', 'deal_changes') }}
        where changed_field_key = 'add_time'
        group by 1
    ) as creation
        on activity.deal_id = creation.deal_id

)

select 'deal ids in deal_changes' as metric, count(*)::text as value from deal_ids
union all select 'deal ids in activity', count(*)::text from call_deal_ids
union all select 'lowest id in any source', lowest_id::text from id_range
union all select 'highest id in any source', highest_id::text from id_range
union all select 'deal_id matches', count(*)::text from call_deal_ids inner join deal_ids using (deal_id)
union all select 'deal_id matches by chance', round((select count(*) from call_deal_ids) * (select count(*) from deal_ids) / id_space, 1)::text from id_range
union all select 'activity_id matches', count(*)::text from activity_ids inner join deal_ids on activity_ids.activity_id = deal_ids.deal_id
union all select 'activity_id matches by chance', round((select count(*) from activity_ids) * (select count(*) from deal_ids) / id_space, 1)::text from id_range
union all select 'linked calls', count(*)::text from overlap_calls
union all select 'linked calls due before the deal was created', count(*)::text from overlap_calls where due_to < created_at
"""
run_analysis(sql)


metric,value
deal ids in deal_changes,1995
deal ids in activity,4572
lowest id in any source,100007
highest id in any source,999964
deal_id matches,8
deal_id matches by chance,10.1
activity_id matches,15
activity_id matches by chance,10.1
linked calls,8
linked calls due before the deal was created,4


10 result rows


**Interpretation.** Only 8 of 4,572 activity deal IDs match the 1,995 deal-history IDs. The uniform-ID-range baseline is approximately 10 matches. That is a rough diagnostic, not proof of synthetic data or a formal significance test. Matching activity IDs instead does not establish a trustworthy relationship either.

**Assumption and decision.** Keep the supplied deal IDs and count stage and call indicators independently. A call KPI counts distinct deal IDs with an accepted completed sales call in its due month, not call events. Do not interpret neighboring stage/call counts as a conversion rate or causal relationship. Partial or synthetic data is a hypothesis; production linkage requires confirmation.

The supplementary `sql/call_user_link_test.sql` also checks user-based alternatives; it is not used to manufacture missing deal links.

## 4. Are stage histories consistent with simple funnel counting?

**Scope:** Raw vs accepted data; requires core models. SQL: `sql/stage_data_quality.sql`.

In [5]:
sql = r"""
-- Question: how often do deals skip stages, move backward, or get a lost reason after Implementation?
-- Backs: the appendix of docs/methodology_and_assumptions.md and section 6 of docs/technical_documentation.md.
-- Result on this extract: 2,502 of 6,911 raw moves skip a stage. All 10 backward or repeated moves sit on excluded deals.
with moves as (

    select
        'accepted' as dataset,
        deal_id,
        new_value::integer as to_stage,
        lag(new_value::integer) over (partition by deal_id order by changed_at, new_value::integer) as from_stage
    from {{ ref('fct_deal_change') }}
    where changed_field = 'stage_id'
        and dq_reason is null

    union all

    select
        'raw',
        deal_id,
        new_value::integer,
        lag(new_value::integer) over (partition by deal_id order by changed_at, new_value::integer)
    from {{ ref('ods_pipedrive__deal_changes') }}
    where changed_field = 'stage_id'

),

lost_after_implementation as (

    select distinct dim_deal.deal_id
    from {{ ref('dim_deal') }} as dim_deal
    inner join {{ ref('fct_deal_stage_entry') }} as entries
        on dim_deal.deal_id = entries.deal_id
        and entries.stage_id = 7
    where dim_deal.lost_at >= entries.entered_at

),

sales_calls as (

    select *
    from {{ ref('fct_activity') }}
    where dq_reason is null
        and activity_type in ('meeting', 'sc_2')

),

sales_call_order as (

    select
        second_call.activity_id,
        exists (
            select 1
            from sales_calls as first_call
            where first_call.deal_id = second_call.deal_id
                and first_call.activity_type = 'meeting'
                and first_call.due_at < second_call.due_at
        ) as has_earlier_call_1
    from sales_calls as second_call
    where second_call.activity_type = 'sc_2'

)

select dataset || ' stage moves' as metric, count(*)::text as value from moves where from_stage is not null group by dataset
union all select dataset || ' skipped moves', count(*)::text from moves where to_stage > from_stage + 1 group by dataset
union all select dataset || ' deals with a skipped move', count(distinct deal_id)::text from moves where to_stage > from_stage + 1 group by dataset
union all select dataset || ' backward or repeated moves', count(*)::text from moves where to_stage <= from_stage group by dataset
union all select 'accepted deals', count(*)::text from {{ ref('dim_deal') }}
union all select 'accepted deals with a lost reason', count(*)::text from {{ ref('dim_deal') }} where deal_status = 'lost'
union all select 'deals lost after entering Implementation', count(*)::text from lost_after_implementation
union all select 'lost reason: ' || lost_reason, count(*)::text from {{ ref('dim_deal') }} group by lost_reason
union all select 'Sales Call 2 rows', count(*)::text from sales_call_order
union all select 'Sales Call 2 rows with no earlier Sales Call 1', count(*)::text from sales_call_order where not has_earlier_call_1
"""
run_analysis(sql)


metric,value
raw stage moves,6911
accepted stage moves,6870
raw skipped moves,2502
accepted skipped moves,2489
accepted deals with a skipped move,1679
raw deals with a skipped move,1684
raw backward or repeated moves,10
accepted deals,1990
accepted deals with a lost reason,1990
deals lost after entering Implementation,1076


17 result rows


**Interpretation.** There are 2,502 skipped transitions among 6,911 raw moves. All 10 backward/repeated moves occur on the deal identities excluded for conflicting creation histories. Lost reasons also appear after Implementation; reaching that stage cannot safely be labeled a win.

**Assumption and decision.** For a forward move from stage 2 to 5, count stages 3, 4 and 5 at the recorded transition time. Mark inferred entries explicitly. A first observation at stage 5 does not invent earlier stages. This supports consistent stage-reach reporting, but does not prove that negotiation or other work actually occurred. Derive loss time from the earliest recorded lost reason and disclose that proxy.

## 5. What does excluding broken identities remove?

**Scope:** Model-dependent quality validation. SQL: `sql/exclusion_impact.sql`.

In [6]:
sql = r"""
-- Question: what does each exclusion remove, and what does it do to the report?
-- Backs: section 6 of docs/technical_documentation.md.
-- Result on this extract: 102 rows for review, 80 from 5 deals and 22 from 11 activity IDs.
with review as (

    select * from {{ ref('rep_records_for_review') }}

),

sales_calls as (

    select *
    from {{ ref('fct_activity') }}
    where activity_type in ('meeting', 'sc_2')

)

select 'rows for review' as metric, count(*)::text as value from review
union all select 'deal_changes rows excluded', count(*)::text from review where source_table = 'deal_changes'
union all select 'deals excluded', count(distinct deal_id)::text from review where source_table = 'deal_changes'
union all select 'activity rows excluded', count(*)::text from review where source_table = 'activity'
union all select 'activity ids excluded', count(distinct record_id)::text from review where source_table = 'activity'
union all select 'done sales calls excluded', count(*)::text from sales_calls where dq_reason is not null and is_done
union all select 'accepted sales call rows', count(*)::text from sales_calls where dq_reason is null
union all select 'accepted done sales calls', count(*)::text from sales_calls where dq_reason is null and is_done
union all select 'accepted activities with a deal history', count(*)::text from {{ ref('fct_activity') }} where dq_reason is null and has_deal_history
union all select 'accepted done sales calls with a deal history', count(*)::text from sales_calls where dq_reason is null and is_done and has_deal_history
union all select 'report rows', count(*)::text from {{ ref('rep_sales_funnel_monthly') }}
union all select 'report zero rows', count(*)::text from {{ ref('rep_sales_funnel_monthly') }} where deals_count = 0
union all select 'report total', sum(deals_count)::text from {{ ref('rep_sales_funnel_monthly') }}
"""
run_analysis(sql)


metric,value
rows for review,102
deal_changes rows excluded,80
deals excluded,5
activity rows excluded,22
activity ids excluded,11
done sales calls excluded,5
accepted sales call rows,2255
accepted done sales calls,1123
accepted activities with a deal history,8
accepted done sales calls with a deal history,2


13 result rows


**Interpretation.** Five deal IDs have conflicting creation histories (80 rows), and 11 activity IDs conflict across 22 rows. The 102 rows remain available for review. There is no evidence that choosing the latest conflicting record would identify the authoritative version.

**Decision and limitation.** Exclude those identities from accepted facts, preserve them with reasons, and measure the effect. A small aggregate exclusion can still bias a subgroup; the remaining report is not guaranteed perfectly accurate. Missing deal-history linkage alone is informational, not a reason to reject an otherwise valid activity. The query's `report total` is a sum of report cells for comparison, never a count of unique deals.

## 6. Is deal creation interchangeable with entry into stage 1?

**Scope:** Model-dependent timing validation. SQL: `sql/stage_timing.sql`.

In [7]:
sql = r"""
-- Question: how long after deal creation is a stage first recorded?
-- Backs: assumption 6 in docs/methodology_and_assumptions.md and the data problems in docs/technical_documentation.md.
-- Result on this extract: a median 35 days to stage 1.
with first_entries as (

    select deal_id, stage_id, min(entered_at) as entered_at
    from {{ ref('fct_deal_stage_entry') }}
    where not is_skipped_stage
    group by 1, 2

)

select
    'median days from creation to stage ' || first_entries.stage_id as metric,
    percentile_cont(0.5) within group (
        order by first_entries.entered_at::date - dim_deal.deal_created_at::date
    )::text as value
from first_entries
inner join {{ ref('dim_deal') }} as dim_deal
    on first_entries.deal_id = dim_deal.deal_id
where first_entries.stage_id <= 3
group by first_entries.stage_id

union all

select 'deals with stage 1 later than creation', count(*)::text
from {{ ref('dim_deal') }}
where first_stage_entered_at > deal_created_at

union all

select 'deals with stage 1 in a later month than creation', count(*)::text
from {{ ref('dim_deal') }}
where date_trunc('month', first_stage_entered_at) > date_trunc('month', deal_created_at)

union all

select 'deal month ends with no stage yet', count(*)::text
from {{ ref('dim_deal') }} as dim_deal
cross join {{ ref('dim_month') }} as dim_month
where dim_deal.deal_created_at::date <= dim_month.month_end
    and dim_deal.first_stage_entered_at::date > dim_month.month_end
    and (dim_deal.lost_at is null or dim_deal.lost_at::date > dim_month.month_end)

order by metric
"""
run_analysis(sql)


metric,value
deal month ends with no stage yet,2274
deals with stage 1 in a later month than creation,1729
deals with stage 1 later than creation,1990
median days from creation to stage 1,35
median days from creation to stage 2,50
median days from creation to stage 3,62


6 result rows


**Interpretation.** The median gap from creation to recorded stage 1 is 35 days. The data does not explain the gap.

**Assumption and decision.** Date Lead Generation by recorded stage-1 entry, rather than creation. Do not infer a stage before the first observation. Some created deals therefore have no known month-end stage yet. Treating the gap as a real waiting period is an alternative requiring business confirmation.

## 7. What does each possible report reading actually count?

**Scope:** Model-dependent definition comparison. SQL: `sql/reading_comparison.sql`.

In [8]:
sql = r"""
-- Question: what would each reading of "deals at a step" give, stage by stage?
-- Backs: the readings table in docs/technical_documentation.md, section 9.
-- Result on this extract: 13,003 movement, 8,860 recorded stage changes, 5,517 last stage, 4,539 month-end positions.
with stage_steps as (

    select stage_id, kpi_name, funnel_step, step_order
    from {{ ref('funnel_steps') }}
    where stage_id is not null

),

movement as (

    select funnel_step, sum(deals_count) as deals
    from {{ ref('rep_sales_funnel_monthly') }}
    group by 1

),

recorded as (

    select stage_id, count(*) as deals
    from (
        select distinct deal_id, stage_id, entered_month
        from {{ ref('fct_deal_stage_entry') }}
        where not is_skipped_stage
    ) as recorded_entries
    group by 1

),

last_stage as (

    select stage_id, count(*) as deals
    from (
        select distinct on (deal_id, entered_month) stage_id
        from {{ ref('fct_deal_stage_entry') }}
        order by deal_id, entered_month, entered_at desc, stage_id desc
    ) as last_entries
    group by 1

),

month_end as (

    select funnel_step, sum(deals_count) as deals
    from {{ ref('rep_deal_stage_distribution_monthly') }}
    group by 1

),

readings as (

    select
        stage_steps.step_order,
        stage_steps.kpi_name,
        movement.deals as movement,
        recorded.deals as recorded_stage_changes,
        last_stage.deals as last_stage_each_month,
        month_end.deals as month_end_distribution
    from stage_steps
    inner join movement on stage_steps.funnel_step = movement.funnel_step
    inner join recorded on stage_steps.stage_id = recorded.stage_id
    inner join last_stage on stage_steps.stage_id = last_stage.stage_id
    inner join month_end on stage_steps.funnel_step = month_end.funnel_step

)

select kpi_name, movement, recorded_stage_changes, last_stage_each_month, month_end_distribution
from readings

union all

select 'Total', sum(movement), sum(recorded_stage_changes), sum(last_stage_each_month), sum(month_end_distribution)
from readings

order by movement desc
"""
run_analysis(sql)


kpi_name,movement,recorded_stage_changes,last_stage_each_month,month_end_distribution
Total,13003,8860,5517,4539
Lead Generation,1990,1990,995,992
Qualified Lead,1985,1474,776,759
Needs Assessment,1946,1300,713,669
Proposal/Quote Preparation,1857,1082,662,593
Negotiation,1688,892,590,456
Closing,1419,738,551,370
Implementation/Onboarding,1076,586,468,297
Follow-up/Customer Success,720,476,440,245
Renewal/Expansion,322,322,322,158


10 result rows


**Interpretation.** Summed comparison cells give 13,003 stage-reach counts, 8,860 recorded-entry counts, 5,517 last-stage-in-month counts and 4,539 month-end positions. These are different measures, not competing estimates of one unique-deal total.

**Decision.** Keep movement as the main report and month-end position as a separate report. Movement counts are distinct within a stage/month cell; the same deal may appear across cells. Month-end positions can be summed across stages within one month, but not across months to obtain unique deals.

## 8. How sensitive are the outputs to the assumptions?

**Scope:** Model-dependent sensitivity analysis. SQL: `sql/assumption_sensitivity.sql`.

In [9]:
sql = r"""
-- Question: how much does each assumption change the numbers, against each alternative not chosen?
-- Backs: the sensitivity table in docs/technical_documentation.md, section 9.
-- Numbers refer to the assumptions in docs/methodology_and_assumptions.md. Sums measure the difference only:
-- they aren't reported KPIs. Movement and position are two measures, so that row is no error or uplift.
with stage_steps as (

    select stage_id, funnel_step
    from {{ ref('funnel_steps') }}
    where stage_id is not null

),

call_steps as (

    select dim_activity_type.activity_type, funnel_steps.funnel_step
    from {{ ref('dim_activity_type') }} as dim_activity_type
    inner join {{ ref('funnel_steps') }} as funnel_steps
        on dim_activity_type.activity_type_name = funnel_steps.kpi_name

),

report as (

    select
        report.month,
        report.funnel_step,
        report.deals_count,
        case when funnel_steps.stage_id is null then 'call' else 'stage' end as population
    from {{ ref('rep_sales_funnel_monthly') }} as report
    inner join {{ ref('funnel_steps') }} as funnel_steps
        on report.funnel_step = funnel_steps.funnel_step

),

month_end_report as (

    select month, funnel_step, deals_count
    from {{ ref('rep_deal_stage_distribution_monthly') }}

),

stage_entries as (

    select * from {{ ref('fct_deal_stage_entry') }}

),

sales_calls as (

    select fct_activity.*, call_steps.funnel_step
    from {{ ref('fct_activity') }} as fct_activity
    inner join call_steps
        on fct_activity.activity_type = call_steps.activity_type

),

-- Assumption 3: the same movement logic over rows the report excludes.
all_stage_moves as (

    select
        deal_id,
        new_value::integer as stage_id,
        changed_at,
        lag(new_value::integer) over (partition by deal_id order by changed_at, new_value::integer) as previous_stage_id
    from {{ ref('fct_deal_change') }}
    where changed_field = 'stage_id'

),

latest_creations as (

    select deal_id, max(changed_at) as created_at
    from {{ ref('fct_deal_change') }}
    where changed_field = 'add_time'
    group by deal_id

),

moves_after_latest_creation as (

    select
        moves.deal_id,
        moves.stage_id,
        moves.changed_at,
        lag(moves.stage_id) over (partition by moves.deal_id order by moves.changed_at, moves.stage_id) as previous_stage_id
    from all_stage_moves as moves
    inner join latest_creations
        on moves.deal_id = latest_creations.deal_id
    where moves.changed_at >= latest_creations.created_at
        or moves.deal_id not in (
            select deal_id from {{ ref('fct_deal_change') }} where dq_reason is not null
        )

),

deals_lost_after_implementation as (

    select dim_deal.deal_id
    from {{ ref('dim_deal') }} as dim_deal
    inner join stage_entries
        on dim_deal.deal_id = stage_entries.deal_id
        and stage_entries.stage_id = 7
    where dim_deal.lost_at >= stage_entries.entered_at

),

sales_call_2_without_call_1 as (

    select second_call.activity_id
    from sales_calls as second_call
    left join sales_calls as first_call
        on second_call.deal_id = first_call.deal_id
        and first_call.activity_type = 'meeting'
        and first_call.due_at < second_call.due_at
    where second_call.activity_type = 'sc_2'
        and first_call.activity_id is null

),

earliest_row_per_activity_id as (

    select distinct on (activity_id) *
    from {{ ref('fct_activity') }}
    order by activity_id, due_at

),

one_row_per_activity_id as (

    select earliest_row.*, call_steps.funnel_step
    from earliest_row_per_activity_id as earliest_row
    inner join call_steps
        on earliest_row.activity_type = call_steps.activity_type

),

months_with_deals as (

    select
        dim_month.month,
        dim_month.month_end,
        dim_deal.deal_id,
        dim_deal.first_stage_entered_at,
        dim_deal.lost_at,
        dim_deal.deal_created_at
    from {{ ref('dim_month') }} as dim_month
    cross join {{ ref('dim_deal') }} as dim_deal

),

extended_months as (

    select month_start::date as month
    from generate_series(
        (select max(month) + interval '1 month' from report),
        (
            select date_trunc('month', max(changed_at))
            from {{ ref('fct_deal_change') }}
            where changed_field = 'lost_reason'
        ),
        interval '1 month'
    ) as month_start

),

alternatives as (

    select 'Key' as assumption, 'Only calls linked to a deal' as alternative, 'call' as population,
        due_month as month, funnel_step, count(distinct deal_id) as deals_count
    from sales_calls
    where dq_reason is null and is_done and has_deal_history
    group by 4, 5

    union all

    select '1', 'Only recorded stage changes', 'stage',
        entered_month, stage_steps.funnel_step, count(distinct deal_id)
    from stage_entries
    inner join stage_steps on stage_entries.stage_id = stage_steps.stage_id
    where not is_skipped_stage
    group by 4, 5

    union all

    select '1', 'Only a deal''s last stage each month', 'stage', month, funnel_step, count(*)
    from (
        select distinct on (deal_id, entered_month)
            entered_month as month,
            stage_steps.funnel_step
        from stage_entries
        inner join stage_steps on stage_entries.stage_id = stage_steps.stage_id
        order by deal_id, entered_month, entered_at desc, stage_entries.stage_id desc
    ) as last_stages
    group by 4, 5

    union all

    select '1', 'Month-end deal / stage distribution (a different measure)', 'stage', month, funnel_step, deals_count
    from month_end_report

    union all

    select '2', 'All scheduled calls', 'call', due_month, funnel_step, count(distinct deal_id)
    from sales_calls
    where dq_reason is null
    group by 4, 5

    union all

    select '3', 'Keeping everything', 'all',
        date_trunc('month', moves.changed_at)::date, stage_steps.funnel_step, count(distinct moves.deal_id)
    from all_stage_moves as moves
    cross join lateral generate_series(
        case when moves.stage_id > moves.previous_stage_id then moves.previous_stage_id + 1 else moves.stage_id end,
        moves.stage_id
    ) as filled(stage_id)
    inner join stage_steps on filled.stage_id = stage_steps.stage_id
    group by 4, 5

    union all

    select '3', 'Keeping everything', 'all', due_month, funnel_step, count(distinct deal_id)
    from sales_calls
    where is_done
    group by 4, 5

    union all

    select '3', 'Keeping one of the conflicting records', 'all',
        date_trunc('month', moves.changed_at)::date, stage_steps.funnel_step, count(distinct moves.deal_id)
    from moves_after_latest_creation as moves
    cross join lateral generate_series(
        case when moves.stage_id > moves.previous_stage_id then moves.previous_stage_id + 1 else moves.stage_id end,
        moves.stage_id
    ) as filled(stage_id)
    inner join stage_steps on filled.stage_id = stage_steps.stage_id
    group by 4, 5

    union all

    select '3', 'Keeping one of the conflicting records', 'all', due_month, funnel_step, count(distinct deal_id)
    from one_row_per_activity_id
    where is_done
    group by 4, 5

    union all

    select '3', 'Removing unusual records', 'all',
        entered_month, stage_steps.funnel_step, count(distinct deal_id)
    from stage_entries
    inner join stage_steps on stage_entries.stage_id = stage_steps.stage_id
    where deal_id not in (select deal_id from deals_lost_after_implementation)
    group by 4, 5

    union all

    select '3', 'Removing unusual records', 'all', due_month, funnel_step, count(distinct deal_id)
    from sales_calls
    where dq_reason is null
        and is_done
        and activity_id not in (select activity_id from sales_call_2_without_call_1)
    group by 4, 5

    union all

    -- One way to guess: a deal that reached Implementation counts as won, so it stops being open.
    select '4', 'Guessing won deals from how far they got', 'month_end', month, funnel_step, deals_count
    from month_end_report
    inner join stage_steps using (funnel_step)
    where stage_steps.stage_id < 7

    union all

    select '5', 'Fixed dates: calendar 2024', 'all', month, funnel_step, deals_count
    from report
    where month < date '2025-01-01'

    union all

    select '5', 'Running to the last lost reason', 'all', month, funnel_step, deals_count
    from report

    union all

    select '5', 'Running to the last lost reason', 'all', extended_months.month, funnel_steps.funnel_step, 0
    from extended_months
    cross join {{ ref('funnel_steps') }} as funnel_steps

    union all

    select '5', 'Leaving out empty months', 'all', month, funnel_step, deals_count
    from report
    where deals_count > 0

    union all

    select '6', 'Showing Deal Created as a step', 'all', month, funnel_step, deals_count
    from report

    union all

    select '6', 'Showing Deal Created as a step', 'all',
        date_trunc('month', deal_created_at)::date, 'Deal Created', count(*)
    from {{ ref('dim_deal') }}
    group by 4, 5

    union all

    select '6', 'Dating Lead Generation by deal creation', 'step_1',
        date_trunc('month', deal_created_at)::date, 'Step 1', count(*)
    from {{ ref('dim_deal') }}
    group by 4, 5

    union all

    select '6', 'Treating the gap as real: a No stage yet row at month end', 'month_end', month, funnel_step, deals_count
    from month_end_report

    union all

    select '6', 'Treating the gap as real: a No stage yet row at month end', 'month_end', month, 'No stage yet', count(*)
    from months_with_deals
    where deal_created_at::date <= month_end
        and first_stage_entered_at::date > month_end
        and (lost_at is null or lost_at::date > month_end)
    group by 4, 5

),

scopes as (

    select distinct assumption, alternative, population
    from alternatives

),

chosen as (

    select scopes.assumption, scopes.alternative, report.month, report.funnel_step, report.deals_count
    from scopes
    inner join report
        on scopes.population = report.population
        or scopes.population = 'all'
        or (scopes.population = 'step_1' and report.funnel_step = 'Step 1')

    union all

    select scopes.assumption, scopes.alternative, month_end_report.month, month_end_report.funnel_step, month_end_report.deals_count
    from scopes
    cross join month_end_report
    where scopes.population = 'month_end'

),

alternative_cells as (

    select assumption, alternative, month, funnel_step, sum(deals_count) as deals_count
    from alternatives
    group by 1, 2, 3, 4

),

compared as (

    select
        coalesce(chosen.assumption, alternative_cells.assumption) as assumption,
        coalesce(chosen.alternative, alternative_cells.alternative) as alternative,
        chosen.deals_count as chosen_count,
        alternative_cells.deals_count as alternative_count
    from chosen
    full join alternative_cells
        on chosen.assumption = alternative_cells.assumption
        and chosen.alternative = alternative_cells.alternative
        and chosen.month = alternative_cells.month
        and chosen.funnel_step = alternative_cells.funnel_step

)

select
    assumption,
    alternative,
    coalesce(sum(chosen_count), 0) as sum_chosen,
    coalesce(sum(alternative_count), 0) as sum_alternative,
    count(chosen_count) as cells_chosen,
    count(alternative_count) as cells_alternative,
    count(*) filter (where coalesce(chosen_count, 0) <> coalesce(alternative_count, 0)) as cells_changed,
    max(abs(coalesce(chosen_count, 0) - coalesce(alternative_count, 0))) as largest_cell_change
from compared
group by 1, 2
order by 1, 2
"""
run_analysis(sql)


assumption,alternative,sum_chosen,sum_alternative,cells_chosen,cells_alternative,cells_changed,largest_cell_change
1,Month-end deal / stage distribution (a different measure),13003,4539,126,126,110,173
1,Only a deal's last stage each month,13003,5517,126,108,99,161
1,Only recorded stage changes,13003,8860,126,110,87,118
2,All scheduled calls,1123,2255,28,18,18,77
3,Keeping everything,14126,14198,154,131,44,4
3,Keeping one of the conflicting records,14126,14180,154,131,34,3
3,Removing unusual records,14126,4995,154,80,122,155
4,Guessing won deals from how far they got,4539,3839,126,84,38,50
5,Fixed dates: calendar 2024,14126,14034,154,132,11,19
5,Leaving out empty months,14126,14126,154,131,0,0


15 result rows


**Interpretation.** The table compares each chosen definition with a named alternative, reporting changed cells and the largest cell difference. The assumption numbers refer to `docs/methodology_and_assumptions.md`. `sum_chosen` and `sum_alternative` summarize comparison cells only; they are not report KPIs. Movement and position answer different questions, so their difference is not uplift or error.

**Decision.** Document the choices with their effects, rather than hide alternatives. The largest practical questions are stage-reach semantics, independent call counting, completion/date rules and reporting coverage. These comparisons validate implementation consequences; shared source defects or incorrect business definitions can survive agreement between calculations.

## 9. Can ownership history support historical attribution?

**Scope:** Model-dependent ownership validation. SQL: `sql/owner_history.sql`.

In [10]:
sql = r"""
-- Question: how many deals changed owner, and did any change fall after a deal's first stage?
-- Backs: the owner limits in docs/technical_documentation.md.
-- Result on this extract: most deals keep one owner, none has more than two, and no owner changes after the first
-- stage, so the last owner also held every stage.
with deal_owners as (

    select
        deal_id,
        count(*) as owner_count,
        max(valid_from) as last_owner_from
    from {{ ref('dim_deal_owner') }}
    group by 1

)

select
    deal_owners.owner_count,
    count(*) as deals,
    count(*) filter (where deal_owners.last_owner_from > dim_deal.first_stage_entered_at) as changed_after_first_stage
from deal_owners
inner join {{ ref('dim_deal') }} as dim_deal
    on deal_owners.deal_id = dim_deal.deal_id
group by 1
order by 1
"""
run_analysis(sql)


owner_count,deals,changed_after_first_stage
1,1494,0
2,496,0


2 result rows


**Interpretation.** 1,494 accepted deals have one owner and 496 have two. No recorded owner change falls after the first observed stage in this extract.

**Decision and limitation.** Keep SCD type 2 deal-owner periods and a last-owner lookup. Historical attribution should join an event date to the ownership interval; using the latest owner alone would rewrite history if later owner changes occurred. User-profile history is unavailable from a single-version users extract. Revenue or causal attribution is outside this report's scope.

## What this analysis established

The main design follows explicit grains, visible assumptions and source-specific coverage. Broken identities are quarantined; weak cross-source linkage does not silently erase calls. Movement, month-end position and last ownership remain separate products of the same reusable models.

This notebook explains the evidence behind the choices. It does not replace dbt tests or prove business completeness. Independent recounts, failure-injection checks and publication limits are documented in `docs/technical_documentation.md`; the initial exploration and later validation are intentionally distinguished here.


In [11]:
conn.rollback()
conn.close()
print("Read-only analysis connection closed.")


Read-only analysis connection closed.


AI tools full disclosure: Codex and Claude Code were used.
